# Chapter 9 — Clinical Literature and PICO (v2026)

> **LangChain 1.x / 2026 refresh.** Pinned versions, optional LangSmith tracing, and reproducible synthetic data. **Research-support only; synthetic/de-identified data; no clinical decisions.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2009.%20LangChain%20for%20Medicine%20and%20Healthcare/LC4LSH_Chapter_9_Clinical_Literature_and_PICO.ipynb)

Extract **PICO** elements (Population, Intervention, Comparison, Outcome) from clinical abstracts using a Pydantic schema, build evidence tables, draft trial-eligibility criteria, and surface study limitations. An optional gated LLM can draft the extraction with **abstention** when evidence is missing.

**Learning objectives**
- Define a PICO schema with Pydantic for structured extraction.
- Build an intervention/outcome evidence table across studies.
- Draft trial eligibility criteria and record study limitations.
- Optionally use an LLM that abstains when the abstract lacks an element.

> **Runtime / cost / data.** Runs locally by default. Optional LLM cells are gated and can be skipped. **Synthetic / de-identified data only. Not for diagnosis, triage, treatment, dosage, prescribing, final billing/coding, EHR writes, or patient messaging.**


## Environment setup

Standard preamble so every chapter notebook starts the same way.


### Secrets

Keys are read from Colab Secrets if available, else from a local `.env`. All optional — the notebook runs without them (LLM cells are skipped).


In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore

    def get_secret(name, default=""):
        return userdata.get(name) or default
except Exception:
    try:
        from dotenv import load_dotenv  # type: ignore

        load_dotenv()
    except Exception:
        pass

    def get_secret(name, default=""):
        return os.environ.get(name, default)


OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY", "")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("OpenAI key set:", bool(OPENAI_API_KEY))


### Install pinned dependencies

Pinned versions keep the notebook reproducible. See `UPDATE_2026.md`.


In [ ]:
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4" "pydantic>=2.5" "pandas" "matplotlib"


### Optional LangSmith tracing

Set `LANGCHAIN_API_KEY` to enable tracing of any LLM calls.


In [ ]:
import os

LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter9-clinical-literature-pico"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")


## PICO schema

Structured output target. Every field is optional so the model can **abstain** (`None`) when the abstract does not state an element.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field


class PICO(BaseModel):
    population: Optional[str] = Field(None, description="Who was studied (condition, demographics).")
    intervention: Optional[str] = Field(None, description="The treatment/exposure being tested.")
    comparison: Optional[str] = Field(None, description="The control or comparator.")
    outcome: Optional[str] = Field(None, description="Primary measured outcome(s).")
    study_type: Optional[str] = Field(None, description="e.g. RCT, cohort, case-control.")
    limitations: Optional[str] = Field(None, description="Stated or inferred limitations.")


print(list(PICO.model_json_schema()["properties"].keys()))


## Synthetic abstracts

Three short synthetic study abstracts (no real patient data).

In [ ]:
ABSTRACTS = [
    {"id": "S1", "text": ("In a randomized controlled trial of 240 adults with stage 1 hypertension, "
        "the SGLT2 inhibitor empagliflozin 10 mg daily was compared with placebo over 12 weeks. "
        "The primary outcome was change in systolic blood pressure. Limitations include short "
        "follow-up and a single-center design.")},
    {"id": "S2", "text": ("A retrospective cohort of older adults with chronic kidney disease examined whether "
        "SGLT2 inhibitor use (vs no use) was associated with slower eGFR decline. The analysis "
        "did not adjust for all confounders.")},
    {"id": "S3", "text": ("This narrative review discusses mechanisms of SGLT2 inhibitors in cardiorenal disease. "
        "No primary data were collected.")},
]
print("abstracts:", len(ABSTRACTS))


### Rule-based PICO extraction (offline baseline)

A transparent keyword baseline that requires **no LLM** and abstains when a cue is absent.

In [ ]:
import re

CUES = {
    "study_type": [("randomized controlled trial", "RCT"), ("rct", "RCT"),
                   ("retrospective cohort", "cohort"), ("cohort", "cohort"),
                   ("case-control", "case-control"), ("review", "narrative review")],
    "intervention": ["sglt2 inhibitor", "empagliflozin", "dapagliflozin"],
    "comparison": ["placebo", "no use", "standard care", "usual care"],
}


def extract_pico(text: str) -> PICO:
    low = text.lower()
    study_type = next((lbl for k, lbl in CUES["study_type"] if k in low), None)
    intervention = next((x for x in CUES["intervention"] if x in low), None)
    comparison = next((x for x in CUES["comparison"] if x in low), None)
    outcome = None
    m = re.search(r"primary outcome was ([^.]+)", low)
    if m:
        outcome = m.group(1).strip()
    population = None
    m = re.search(r"(?:adults|patients) with ([^,.]+)", low)
    if m:
        population = "adults with " + m.group(1).strip()
    limitations = None
    m = re.search(r"limitations? (?:include|was|is)?[:\s]+([^.]+)", low)
    if m:
        limitations = m.group(1).strip()
    return PICO(population=population, intervention=intervention, comparison=comparison,
                outcome=outcome, study_type=study_type, limitations=limitations)


for a in ABSTRACTS:
    print(a["id"], "->", extract_pico(a["text"]).model_dump())


### Evidence table

One row per study with its extracted PICO elements.

In [ ]:
import pandas as pd

rows = []
for a in ABSTRACTS:
    d = extract_pico(a["text"]).model_dump()
    d["id"] = a["id"]
    rows.append(d)

df = pd.DataFrame(rows).set_index("id")
df.fillna("-")


### Trial eligibility draft (from Population)

A naive inclusion/exclusion sketch derived from the population description. **Illustrative only.**

In [ ]:
def draft_eligibility(p: PICO) -> dict:
    if not p.population:
        return {"inclusion": [], "exclusion": [], "note": "abstain - population not stated"}
    return {
        "inclusion": [f"Adults with {p.population.replace('adults with ', '')}"],
        "exclusion": ["Pregnancy", "eGFR < 30 (safety)", "inability to consent"],
        "note": "DRAFT - requires clinician review",
    }


print(draft_eligibility(extract_pico(ABSTRACTS[0]["text"])))
print(draft_eligibility(extract_pico(ABSTRACTS[2]["text"])))


### Optional: LLM extraction with abstention

Gated behind the OpenAI key. Uses structured output so the model returns the PICO schema and may set fields to `null` when absent.

In [ ]:
import os

if os.environ.get("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    structured = llm.with_structured_output(PICO)
    res = structured.invoke(
        "Extract PICO elements. Use null for any element not stated.\n\n" + ABSTRACTS[1]["text"]
    )
    print(res.model_dump())
else:
    print("OPENAI_API_KEY not set - skipping LLM extraction (offline baseline above is sufficient).")


## Limitations & safety

- **Research-support only.** Not for diagnosis, triage, treatment recommendation, dosage, prescribing, final billing/coding, EHR writes, or patient messaging.
- **Synthetic / de-identified data only.** Real PHI requires governance, BAA-covered infrastructure, and access controls.
- Optional LLM outputs are **drafts for human review** and can hallucinate — always verify against source data.
- Any scoring/eligibility logic here is illustrative and must be validated by qualified clinicians before any real use.


In [ ]:
# Cleanup: drop references and free memory.
import gc

for _name in ["llm", "chain", "model"]:
    globals().pop(_name, None)

gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Q1. Why make every PICO field optional in the schema?</summary>
So the extractor can abstain (`null`) when the abstract does not state an element, instead of hallucinating a value. Abstention is safer for evidence synthesis.
</details>

<details><summary>Q2. Why keep a transparent rule-based baseline even when an LLM is available?</summary>
The keyword baseline is reproducible, free, and auditable. It provides a fallback when no API key is present and a sanity check against LLM hallucination.
</details>

<details><summary>Q3. Why must the trial-eligibility draft be labelled "requires clinician review"?</summary>
Eligibility criteria affect patient safety and trial validity. A naive text-derived draft omits clinical nuance and must be validated by qualified clinicians before use.
</details>

### Task A — Add a sample-size extractor
Parse and record the sample size (e.g. "240 adults") into the schema and evidence table.

### Task B — Confidence per field
Add a per-field confidence (high when a cue matched, low otherwise) and surface low-confidence fields for human review.

### Task C — Comparison across studies
Build a pivot showing which studies share the same intervention and comparison to reveal evidence clusters.

### Task D — Limitations taxonomy
Classify limitations into a small taxonomy (bias, sample size, follow-up, generalizability) and count them across studies.
